# CSC 2301 · Week 1 — Python basics with drill-hole data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W1_python_basics.ipynb)

**IUGB · Bachelor of Science in Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

In this first lab you will use Python the way an engineer does on a real project: read measurements
from a file, convert units, summarise them and make a simple decision. The data are gold assays from
six exploration drill holes (synthetic, but realistic).

**What you will practise:** variables and types · arithmetic · lists and `for` loops · reading a CSV file ·
writing functions · `if` / `elif` / `else` · dictionaries.

**How to work:** run each cell with **Shift + Enter**, from top to bottom. Start with the two setup cells below.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/drillholes_assays.csv', 'drillholes_assays.csv'), ('tests/w1/q1.py', 'tests/q1.py'), ('tests/w1/q2.py', 'tests/q2.py'), ('tests/w1/q3.py', 'tests/q3.py'), ('tests/w1/q4.py', 'tests/q4.py'), ('tests/w1/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. Variables and arithmetic

A variable is a name for a value. Python has a few basic **types**: `int` (whole numbers),
`float` (numbers with decimals), `str` (text) and `bool` (`True` / `False`).

Example: the mass of rock in a block of ore is *volume × density*.

In [ ]:
length_m = 20      # int
width_m = 10
height_m = 5
density = 2.7      # float, tonnes per cubic metre
rock_type = "granite"   # str

volume_m3 = length_m * width_m * height_m
tonnes = volume_m3 * density
print("Volume:", volume_m3, "m3")
print("Mass:", tonnes, "t of", rock_type)
print(type(volume_m3), type(tonnes), type(rock_type))

Gold grades are often reported in **ppb** (parts per billion) by the laboratory, and in **g/t**
(grams per tonne) by mining engineers. 1 g/t = 1000 ppb. If the block above grades 1.8 g/t,
how much gold does it hold?

In [ ]:
grade_gpt = 1.8
gold_g = tonnes * grade_gpt
print(f"Gold in the block: {gold_g:.0f} g, or {gold_g / 31.1035:.1f} troy ounces")

## 2. Lists and loops

A **list** holds several values in order. A `for` loop repeats an action for each value.

In [ ]:
grades = [0.2, 1.4, 3.1, 0.8, 0.05]   # g/t, five samples from one hole

print("First sample:", grades[0])
print("Last sample:", grades[-1])
print("Number of samples:", len(grades))

total = 0
for g in grades:
    total = total + g
print("Mean grade:", total / len(grades))
print("Same with built-ins:", sum(grades) / len(grades), "max:", max(grades))

## 3. Reading the drill-hole file

The file `drillholes_assays.csv` has one line per **assay interval**: a slice of core
between two depths, sent to the lab. The columns are:

| column | meaning |
|---|---|
| `hole_id` | drill hole name, e.g. `DH-01` |
| `from_m`, `to_m` | top and bottom of the interval, in metres |
| `au_ppb` | gold content, in ppb |
| `lithology` | rock type |

Python's `csv` module reads each line as a **dictionary** (`dict`): column name → value.
Everything read from a file is text, so we convert the numbers ourselves.

In [ ]:
import csv

rows = []
with open("drillholes_assays.csv", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append({
            "hole_id": r["hole_id"],
            "from_m": float(r["from_m"]),
            "to_m": float(r["to_m"]),
            "au_ppb": int(r["au_ppb"]),
            "lithology": r["lithology"],
        })

for r in rows[:3]:
    print(r)

## 4. Functions and decisions

A **function** packages a calculation so you can reuse it. `if` / `elif` / `else` lets the program choose.

In [ ]:
def interval_length(row):
    """Length of an assay interval, in metres."""
    return row["to_m"] - row["from_m"]

def describe_depth(depth_m):
    if depth_m < 10:
        return "near surface"
    elif depth_m < 30:
        return "shallow"
    else:
        return "deep"

first = rows[0]
print(first["hole_id"], "interval length:", interval_length(first), "m")
print("Depth 25 m is", describe_depth(25))

---
## Exercises

Write your answer in each cell, then run `grader.check("q…")` below it.
A green message means the check passed; otherwise read the hint and try again.
You can run a check as many times as you like.

### Q1. Counting

How many assay intervals are in the file? Store the answer in `n_intervals` (an `int`). Use `rows`.

In [ ]:
n_intervals = ...
n_intervals

In [ ]:
grader.check("q1")

### Q2. Unit conversion

Write a function `ppb_to_gpt(ppb)` that converts a grade in ppb to g/t (1 g/t = 1000 ppb).

In [ ]:
def ppb_to_gpt(ppb):
    ...

ppb_to_gpt(1850)

In [ ]:
grader.check("q2")

### Q3. Mean grade of one hole

Intervals do not all have the same length, so a plain average is biased. Engineers use the
**length-weighted mean**:

$$\bar g = \frac{\sum_i g_i \, \ell_i}{\sum_i \ell_i}$$

where $g_i$ is the grade in g/t and $\ell_i$ the length of interval *i*.
Compute it for hole `DH-03` and store it in `mean_dh03` (a `float`).
Use a loop, `interval_length` and your `ppb_to_gpt`.

In [ ]:
mean_dh03 = ...
mean_dh03

In [ ]:
grader.check("q3")

### Q4. Ore or waste?

Write a function `classify(grade_gpt)` that returns:

* `"waste"` if the grade is below 0.5 g/t,
* `"low grade"` from 0.5 up to (but not including) 2.0 g/t,
* `"high grade"` from 2.0 g/t.

In [ ]:
def classify(grade_gpt):
    ...

[classify(g) for g in (0.3, 1.0, 4.2)]

In [ ]:
grader.check("q4")

### Q5. Which hole is the richest?

Build a dictionary `hole_means` that maps each `hole_id` to its length-weighted mean grade in g/t
(as in Q3, but for every hole). Then store in `richest_hole` the `hole_id` with the highest mean.

*Hint:* `max(d, key=d.get)` returns the key of a dictionary `d` with the largest value.

In [ ]:
hole_means = {}
...
richest_hole = ...
hole_means, richest_hole

In [ ]:
grader.check("q5")

### Going further (not checked)

* How many intervals of each class (Q4) does each hole contain?
* Is gold richer in one lithology? Compute the mean grade per `lithology`.

---
**Before you leave:** run all the checks at once with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()